# Business entity resolution on SageMaker Studio
Run the cells top to bottom (**Run > Run All Cells** is fine for cells 1-4). No terminal needed.

Before you start, on your laptop: `bash upload_to_s3.sh` (moves the code, the dataset and the finished work to S3).
In Studio: use a **JupyterLab space with a GPU instance (ml.g5.4xlarge or larger), storage of at least 100 GB, and idle shutdown switched off.**

## 1. Settings

In [ ]:
import os, pathlib, subprocess, sys, time
BUCKET = "sagemaker-er-050307929098"   # the S3 bucket used by upload_to_s3.sh (name must start with "sagemaker-")
REGION = "ap-southeast-2"
MODE = "continue"                      # "continue": reuse the finished work uploaded from the laptop | "fresh": recompute everything (4-6 h)
HOME = pathlib.Path.home()
CODE, DATA, WORK = HOME / "er", HOME / "data", HOME / "er_work"
os.environ["AWS_DEFAULT_REGION"] = REGION

def sh(cmd, tail=2500):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print((r.stdout + r.stderr)[-tail:])
    assert r.returncode == 0, f"command failed ({r.returncode}): {cmd}"

## 2. Download the code and the data from S3 (one command per folder, any number of files)

In [ ]:
for d in (CODE, DATA, WORK): d.mkdir(exist_ok=True)
if (CODE / "main.py").exists():
    print("code already in", CODE, "(uploaded by hand): skipping the code download")
else:
    sh(f"aws s3 cp s3://{BUCKET}/er_code.tgz {HOME}/er_code.tgz --only-show-errors && tar xzf {HOME}/er_code.tgz -C {CODE}")
sh(f"aws s3 sync s3://{BUCKET}/dataset {DATA}/dataset --only-show-errors")
sh(f"aws s3 sync s3://{BUCKET}/utils {DATA}/utils --only-show-errors")
if MODE == "continue":
    sh(f"aws s3 sync s3://{BUCKET}/er_work {WORK} --only-show-errors")
sh(f"ls {CODE} | head -40; du -sh {DATA} {WORK}")

## 3. Install the environment (a few minutes) and check the GPU

In [ ]:
PY = CODE / ".venv" / "bin" / "python"
if not PY.exists():
    sh(f"python3 -m venv {CODE}/.venv")
sh(f"{PY} -m pip install -q --upgrade pip")
sh(f"{PY} -m pip install -q torch==2.6.0 --index-url https://download.pytorch.org/whl/cu124")
sh(f"{PY} -m pip install -q -r {CODE}/requirements.txt")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader")
sh(f"{PY} -c \"import torch; print('cuda available:', torch.cuda.is_available())\"")

## 4. See what will run (finished steps are skipped)

In [ ]:
sh(f"cd {CODE} && {PY} main.py --dataset {DATA}/dataset --root {WORK} --list", tail=6000)

## 5. Start the pipeline in the background
It keeps running if you close the browser tab. If it stops for any reason, run this cell again: finished steps are skipped.

In [ ]:
log = open(HOME / "run.log", "a")
proc = subprocess.Popen([str(PY), "-u", "main.py", "--dataset", str(DATA / "dataset"), "--root", str(WORK)],
                        cwd=CODE, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
print("started, pid", proc.pid)

## 6. Watch progress (run this cell as often as you like)

In [ ]:
sh(f"grep -E '^(==|--|!!)|finished in|held-out' {HOME}/run.log | tail -25; echo; tail -n 3 {HOME}/run.log; echo; nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader", tail=5000)

## 7. When it has finished: copy the results to S3
The final files are in `~/er_work/output/`. Download them to your laptop with `aws s3 cp` from Git Bash.

In [ ]:
sh(f"ls -la {WORK}/output; aws s3 cp {WORK}/output s3://{BUCKET}/output --recursive --only-show-errors")

**Stop the space in Studio when you are done** (compute is billed while it runs).